# Early Sample Selection with LT Signals (Section 5.3)

Evaluates LT signals as a tool for selecting the best answer from a fixed set of parallel generations using only a *partial* reasoning trace (Section 5.3, Figure 6, Table 2, Appendix H).

**Setting:** Given N parallel samples, observe only the first K tokens (in 500-token steps) of each trace. Use LT signal values accumulated up to step K to rank the samples and select the one most likely to be correct — without waiting for traces to finish.

The notebook covers:
- **Figure 6** (AUC by step): discriminative power of LT signals as a function of how many 500-token segments have been observed.
- **Table 2** (Classifier selection): a random forest trained on per-step LT features selects among 5 parallel samples at 2 000 tokens (step 4).
- **Wang et al. (2024) comparison**: ST-BoN baseline that uses cross-layer curvature signals to select the sample most similar to the centroid.
- **Appendix H** (LODO): classifier trained on two datasets and evaluated on the third (leave-one-dataset-out cross-dataset generalization).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
import torch

import sys
sys.path.insert(0, '../src')
from post_utils import load_early_exit

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
PATH = Path('../')  # project root — adjust if running from a different location

## Configuration

In [ ]:
MODELS = ['phi4-reasoning-plus', 'deepseek-r1-qwen14B', 'qwen3-14B']
DATASETS = ['GPQA', 'AIME2025', 'TSP']

MODEL_NAME_MAP = {
    'phi4-reasoning-plus': 'Phi4R+',
    'deepseek-r1-qwen14B': 'R1-D',
    'qwen3-14B': 'Qwen3',
}

METRICS = ['net_by_step', 'cumulative_by_step']
LABEL_MAP = {
    'net_by_step':        'Net Change',
    'cumulative_by_step': 'Cumulative Change',
}
NEGATE = {'cumulative_by_step'}  # higher → less confident, invert AUC

## Data loading

In [ ]:
def load_step_vectors_df(model_name, dataset):
    """Build a per-step DataFrame from step-vector internals and early-exit accuracy."""
    early_exit_df = load_early_exit(model_name, dataset, PATH)
    step_vectors  = torch.load(
        PATH / f"results/internals/{dataset}/{model_name}_ntokens_500_step_vectors_residual.pt"
    )

    rows = []
    for i in step_vectors:
        datapoint  = i['data_point_id']
        data_repeat = i['data_repeat_id']

        d_exit = early_exit_df.loc[
            (early_exit_df['data_point_id']  == datapoint) &
            (early_exit_df['data_repeat_id'] == data_repeat)
        ]
        acc           = d_exit.loc[d_exit['percentage'] == 100, 'acc'].item()
        n_think_tokens = i['n_think_tokens']

        cumulative_by_step = torch.cumsum(i['cumulative_by_step'], dim=0)
        steps              = torch.arange(1, len(cumulative_by_step) + 1)
        net_by_step        = i['net_by_step'] / steps.float()

        for s in range(len(steps)):
            rows.append({
                'datapoint':           datapoint,
                'data_repeat':         data_repeat,
                'accuracy':            acc,
                'final_n_think_tokens': n_think_tokens,
                'step':                int(steps[s].item()),
                'net_by_step':         float(net_by_step[s].item()),
                'cumulative_by_step':  float(cumulative_by_step[s].item()),
            })

    return pd.DataFrame(rows)

## AUC by step (Figure 6)

For each reasoning step K (500 tokens each), computes the ROC-AUC of the accumulated LT signal against binary answer correctness. Shows how discriminative power grows as more of the trace is observed. Corresponds to Figure 6 in the paper.

Signals used:
- `net_by_step` (Net Change): L2 norm of the cumulative displacement from segment 1 to segment K, normalised by K.
- `cumulative_by_step` (Cumulative Change): running total of consecutive segment displacements up to step K (higher → less confident; AUC is negated).

In [ ]:
auc_records = []

for model_name in MODELS:
    for dataset in DATASETS:
        print(model_name, dataset)
        df = load_step_vectors_df(model_name, dataset)

        for metric in METRICS:
            for step in df['step'].unique():
                s_df = (df[df['step'] <= step]
                        .groupby(['datapoint', 'data_repeat'])
                        .tail(1))

                y      = s_df['accuracy'].astype(bool).values
                scores = s_df[metric].values
                mask   = np.isfinite(scores)

                auc = np.nan
                if mask.sum() > 0 and np.unique(y[mask]).size == 2:
                    auc = roc_auc_score(y[mask], scores[mask])
                    if metric in NEGATE:
                        auc = 1 - auc

                auc_records.append({
                    'model':   model_name,
                    'dataset': dataset,
                    'metric':  metric,
                    'step':    step,
                    'ntokens': step * 500,
                    'auc':     auc,
                })

auc_df = pd.DataFrame(auc_records)

In [ ]:
auc_df.groupby(['metric', 'step'])['auc'].mean().unstack('metric').round(3)

## Classifier-based sample selection (Table 2)

Trains a Random Forest classifier on the per-step LT features (net and cumulative change at each step up to K) to predict which of the 5 parallel samples is correct, using GroupKFold CV grouped by question. At inference, selects the sample with the highest predicted probability.

Results are shown at step K = 4 (2 000 tokens) and correspond to Table 2 in the paper. Columns:
- `classifier_acc` — fraction of questions where the selected sample is correct
- `saved_tokens_%` — tokens saved vs. running all 5 traces to completion

In [ ]:
def get_meta_model(strategy='rf', normalize=True, seed=42):
    if strategy == 'logreg':
        base = LogisticRegression(max_iter=5000, solver='lbfgs', random_state=seed)
    elif strategy == 'svm':
        base = LinearSVC(max_iter=5000, random_state=seed)
    elif strategy == 'dtree':
        base = DecisionTreeClassifier(max_depth=5, random_state=seed)
    elif strategy == 'rf':
        base = RandomForestClassifier(n_estimators=200, random_state=seed)
    else:
        raise ValueError(f'Unknown strategy: {strategy}')
    if normalize and strategy in ('logreg', 'svm'):
        return make_pipeline(StandardScaler(), base)
    return make_pipeline(base)


def sweep_steps_classifier(internals_df, steps=range(1, 6),
                            feature_cols=None, n_splits=3,
                            classifier_strategy='rf'):
    if feature_cols is None:
        feature_cols = ['net_by_step', 'cumulative_by_step']

    results = []
    for step in steps:
        s_df = (internals_df[internals_df['step'] <= step]
                .groupby(['datapoint', 'data_repeat'])
                .tail(1)
                .reset_index(drop=True))

        groups = s_df['datapoint'].values
        X      = s_df[feature_cols].values
        y      = s_df['accuracy'].astype(int).values

        gkf   = GroupKFold(n_splits=n_splits)
        preds = np.zeros(len(s_df))

        for train_idx, test_idx in gkf.split(X, y, groups=groups):
            clf = get_meta_model(classifier_strategy)
            clf.fit(X[train_idx], y[train_idx])
            if classifier_strategy == 'svm':
                fp = clf.decision_function(X[test_idx])
                fp = (fp - fp.min()) / (fp.max() - fp.min() + 1e-8)
            else:
                fp = clf.predict_proba(X[test_idx])[:, 1]
            preds[test_idx] = fp

        s_df = s_df.copy()
        s_df['pred_proba'] = preds
        top_df = s_df.loc[s_df.groupby('datapoint')['pred_proba'].idxmax()]

        total_tokens    = int(s_df['final_n_think_tokens'].sum())
        selected_tokens = int(top_df['final_n_think_tokens'].sum())
        excluded_tokens = sum(min(t, step * 500) for t in s_df.drop(top_df.index, errors='ignore')['final_n_think_tokens'])
        saved_pct       = ((total_tokens - selected_tokens - excluded_tokens) / total_tokens * 100
                           if total_tokens > 0 else 0.0)

        results.append({
            'step':           step,
            'classifier_acc': top_df['accuracy'].mean(),
            'saved_tokens_%': saved_pct,
        })

    return pd.DataFrame(results)

In [ ]:
across_steps_records = []

for model_name in MODELS:
    for dataset in DATASETS:
        print(f'Model: {MODEL_NAME_MAP[model_name]} | Dataset: {dataset}')
        df = load_step_vectors_df(model_name, dataset)
        results_df = sweep_steps_classifier(df, steps=range(1, 6), classifier_strategy='rf')
        results_df['model']   = model_name
        results_df['dataset'] = dataset
        across_steps_records.append(results_df)

across_steps_df = pd.concat(across_steps_records, ignore_index=True)

In [ ]:
across_steps_df[across_steps_df['step'] == 4].groupby(['model', 'dataset'])[['classifier_acc', 'saved_tokens_%']].mean().round(4)

## Wang et al. (2024) baseline comparison

Reimplements the ST-BoN (Shortest-Trajectory Best-of-N) selection rule from Wang et al. (2024): select the sample whose cross-layer curvature trajectory at step K = 2 000 // n_tokens is closest to the centroid (minimum mean pairwise deviation). Reports accuracy and token savings on the same model × dataset grid as Table 2, providing a direct comparison to the classifier-based LT approach above.

In [ ]:
def compute_wang_results(models, datasets, n_tokens=500):
    n_steps = 2000 // n_tokens

    records = []
    for model_name in models:
        for dataset in datasets:
            internals = torch.load(
                PATH / f"results/internals/{dataset}/{model_name}_ntokens_{n_tokens}_internals.pt"
            )
            early_exit_df = load_early_exit(model_name, dataset, PATH)

            rows = []
            for i in internals:
                mag_change = i['layerwise_mag_changes'] / i['layerwise_mag_overall_change']
                ang_change = i['layerwise_ang_changes'] / i['layerwise_ang_overall_change']
                change = torch.mean(mag_change - ang_change, dim=0)

                acc = early_exit_df[
                    (early_exit_df['data_point_id']  == i['data_point_id']) &
                    (early_exit_df['data_repeat_id'] == i['data_repeat_id']) &
                    (early_exit_df['percentage'] == 100)
                ]['acc'].item()

                rows.append({
                    'datapoint':   i['data_point_id'],
                    'data_repeat': i['data_repeat_id'],
                    'n_tokens':    i['n_think_tokens'],
                    'change':      change,
                    'acc':         acc,
                })

            results_df = pd.DataFrame(rows)

            accs, used_tokens, total_tokens = [], [], []
            for datapoint in results_df['datapoint'].unique():
                sub_df = results_df[results_df['datapoint'] == datapoint]
                values = [
                    row['change'][min(n_steps, len(row['change']) - 1)]
                    for _, row in sub_df.iterrows()
                ]
                diff  = torch.tensor(values)[:, None] - torch.tensor(values)[None, :]
                index = int(torch.argmin(diff.sum(dim=1) / (diff.shape[1] - 1)))

                accs.append(sub_df.iloc[index]['acc'].item())
                total_tokens.append(sub_df['n_tokens'].sum().item())
                sel_tokens = sub_df.iloc[index]['n_tokens'].item()
                used_tokens.append(sel_tokens + 2000 * (len(sub_df) - 1))

            saved_pct = (1 - np.mean(used_tokens) / np.mean(total_tokens)) * 100
            records.append({
                'model':          model_name,
                'dataset':        dataset,
                'classifier_acc': np.mean(accs),
                'saved_tokens_%': saved_pct,
            })

    return pd.DataFrame(records)

In [ ]:
wang_df = compute_wang_results(MODELS, DATASETS)

In [ ]:
wang_df.groupby(['model', 'dataset'])[['classifier_acc', 'saved_tokens_%']].mean().round(4)

## Cross-dataset generalization — LODO (Appendix H)

Leave-One-Dataset-Out (LODO) evaluation: train the Random Forest classifier on all (model, dataset) combinations *except* the target dataset, then evaluate on the held-out dataset. This tests whether LT-based classifiers generalise across tasks without any in-domain labels. Corresponds to Table 9 / Appendix H in the paper.

In [ ]:
def build_all_datasets_df(model_name, datasets):
    dfs = []
    for dataset in datasets:
        df = load_step_vectors_df(model_name, dataset)
        df['dataset'] = dataset
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


def train_classifier_lodo(all_df, eval_dataset, feature_cols, classifier_strategy='rf', n_splits=3):
    """Train on all datasets except eval_dataset (leave-one-dataset-out)."""
    train_df = all_df[all_df['dataset'] != eval_dataset]
    groups   = train_df['datapoint'].values
    X        = train_df[feature_cols].values
    y        = train_df['accuracy'].astype(int).values

    gkf   = GroupKFold(n_splits=n_splits)
    clf   = get_meta_model(classifier_strategy)
    for train_idx, _ in gkf.split(X, y, groups=groups):
        clf.fit(X[train_idx], y[train_idx])
    return clf


def probe_dataset(internals_df, classifier, feature_cols, steps):
    results = []
    for step in steps:
        s_df = (internals_df[internals_df['step'] <= step]
                .groupby(['datapoint', 'data_repeat'])
                .tail(1)
                .reset_index(drop=True))

        X = s_df[feature_cols].values
        if hasattr(classifier, 'predict_proba'):
            preds = classifier.predict_proba(X)[:, 1]
        else:
            preds = classifier.decision_function(X)
            preds = (preds - preds.min()) / (preds.max() - preds.min() + 1e-8)

        s_df = s_df.copy()
        s_df['pred_proba'] = preds
        top_df = s_df.loc[s_df.groupby('datapoint')['pred_proba'].idxmax()]

        total_tokens    = int(s_df['final_n_think_tokens'].sum())
        selected_tokens = int(top_df['final_n_think_tokens'].sum())
        excluded_tokens = sum(min(t, step * 500) for t in s_df.drop(top_df.index, errors='ignore')['final_n_think_tokens'])
        saved_pct = ((total_tokens - selected_tokens - excluded_tokens) / total_tokens * 100
                     if total_tokens > 0 else 0.0)

        results.append({
            'step':           step,
            'classifier_acc': top_df['accuracy'].mean(),
            'saved_tokens_%': saved_pct,
        })
    return pd.DataFrame(results)


def run_lodo_experiment(models, datasets, feature_cols=None, steps=range(1, 6)):
    if feature_cols is None:
        feature_cols = ['net_by_step', 'cumulative_by_step']

    all_results = []
    for model_name in models:
        print(f'\n=== {MODEL_NAME_MAP[model_name]} ===')
        all_df = build_all_datasets_df(model_name, datasets)

        for eval_dataset in datasets:
            print(f'  leave out: {eval_dataset}')
            clf = train_classifier_lodo(all_df, eval_dataset, feature_cols)
            eval_df = load_step_vectors_df(model_name, eval_dataset)
            res = probe_dataset(eval_df, clf, feature_cols, steps)
            res['model']   = model_name
            res['dataset'] = eval_dataset
            all_results.append(res)

    return pd.concat(all_results, ignore_index=True)

In [ ]:
lodo_df = run_lodo_experiment(MODELS, DATASETS, steps=range(1, 5))

In [ ]:
lodo_df.groupby(['model', 'dataset'])[['classifier_acc', 'saved_tokens_%']].mean().round(4)